# Bias and Fairness in AI/ML

## Overview
AI systems can encode, amplify, and perpetuate societal biases. Understanding and measuring bias is a prerequisite for building fair, trustworthy AI systems. This notebook covers the mathematical foundations of fairness and practical tools for bias mitigation.

---

## 1. Sources of Bias

### Data Bias
- **Historical bias**: Data reflects historical discrimination (e.g., hiring data favors men)
- **Representation bias**: Underrepresentation of minority groups
- **Measurement bias**: Different measurement accuracy across groups
- **Aggregation bias**: Using a single model for heterogeneous populations
- **Label bias**: Annotators disagree based on cultural background

### Algorithmic Bias
- Models optimize for average performance, harming minority subgroups
- Proxy discrimination: using ZIP code as a proxy for race
- Feedback loops: biased predictions → biased training data → worse bias

### Deployment Bias
- Model used in contexts different from training
- Different error rates manifest differently in high-stakes settings

---

## 2. Protected Attributes

Legally protected characteristics (vary by jurisdiction):
- Race, ethnicity, national origin
- Gender, sex
- Age
- Disability
- Religion
- Sexual orientation

Even if not used directly, **proxy variables** can encode protected attributes (zip code → race, name → gender).

---

## 3. Fairness Definitions (Mathematical)

Let $Y$ = true label, $\hat{Y}$ = prediction, $A$ = protected attribute (e.g., $A=0$: majority, $A=1$: minority)

### 3.1 Demographic Parity (Statistical Parity)
Positive prediction rates are equal across groups:
$$P(\hat{Y}=1 \mid A=0) = P(\hat{Y}=1 \mid A=1)$$

**Disparate Impact Ratio**: $$DI = \frac{P(\hat{Y}=1 \mid A=1)}{P(\hat{Y}=1 \mid A=0)} \geq 0.8$$ (80% rule in US law)

### 3.2 Equal Opportunity
True positive rates are equal across groups (requires $Y=1$):
$$P(\hat{Y}=1 \mid Y=1, A=0) = P(\hat{Y}=1 \mid Y=1, A=1)$$

### 3.3 Equalized Odds
Both TPR and FPR are equal across groups:
$$P(\hat{Y}=1 \mid Y=y, A=0) = P(\hat{Y}=1 \mid Y=y, A=1) \quad \forall y \in \{0,1\}$$

### 3.4 Predictive Parity (Calibration)
Predicted probabilities are equally calibrated:
$$P(Y=1 \mid \hat{Y}=p, A=0) = P(Y=1 \mid \hat{Y}=p, A=1)$$

### 3.5 Individual Fairness
Similar individuals receive similar predictions:
$$d(x_i, x_j) \leq \epsilon \implies d(f(x_i), f(x_j)) \leq \delta$$

### 3.6 Counterfactual Fairness
Prediction would be the same if protected attribute were different:
$$P(\hat{Y}_{A\leftarrow a} = y \mid X=x, A=a) = P(\hat{Y}_{A\leftarrow a'} = y \mid X=x, A=a)$$

---

## 4. Fairness Impossibility Theorem

**Chouldechova (2017)** proved that when base rates differ between groups, you **cannot simultaneously satisfy**:
1. Predictive parity
2. Equal false positive rates
3. Equal false negative rates

This means choosing a fairness criterion is a **value judgment**, not a technical optimization.

---

## 5. Bias Mitigation Strategies

### Pre-processing (Fix the data)
- **Reweighing**: assign higher weights to underrepresented groups
- **Resampling**: oversample minority, undersample majority
- **Data augmentation**: generate synthetic minority samples

### In-processing (Fix the model)
- **Adversarial debiasing**: train adversary to not predict protected attribute from model output
- **Fairness constraints**: add fairness penalty to loss function:
$$L_{total} = L_{task} + \lambda \cdot L_{fairness}$$
- **Prejudice remover**: regularization term based on mutual information

### Post-processing (Fix the predictions)
- **Threshold optimization**: set different decision thresholds per group
- **Calibrated equalized odds**: adjust predictions to satisfy equalized odds
- **Reject option classification**: abstain on uncertain predictions near boundary

In [1]:
# pip install aif360 fairlearn scikit-learn
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# ── Simulate a biased dataset ───────────────────────────────────
np.random.seed(42)
n = 2000

# Protected attribute: 0 = majority, 1 = minority
A = np.random.binomial(1, 0.3, n)  # 30% minority

# Features (minority group has systematically lower scores due to historical bias)
X1 = np.random.normal(loc=1 - 0.4*A, scale=1, size=n)  # qualification score
X2 = np.random.normal(loc=0.5*(1-A), scale=0.8, size=n)  # experience
X = np.column_stack([X1, X2])

# True label: hired (1 = yes)
Y = (X1 + X2 > 0.3).astype(int)

# Split
X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
    X, Y, A, test_size=0.3, random_state=42
)

# Train a standard logistic regression
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Overall accuracy: {(y_pred == y_test).mean():.3f}")

Overall accuracy: 0.988


In [2]:
# ── Fairness Metrics from Scratch ──────────────────────────────
def fairness_metrics(y_true, y_pred, protected):
    """Compute key fairness metrics by group."""
    results = {}
    for group in [0, 1]:
        mask = (protected == group)
        yt, yp = y_true[mask], y_pred[mask]
        tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()
        results[group] = {
            'n': mask.sum(),
            'positive_rate': yp.mean(),
            'tpr': tp / (tp + fn) if (tp + fn) > 0 else 0,
            'fpr': fp / (fp + tn) if (fp + tn) > 0 else 0,
            'precision': tp / (tp + fp) if (tp + fp) > 0 else 0,
            'accuracy': (tp + tn) / len(yt)
        }
    return results

metrics = fairness_metrics(y_test, y_pred, A_test)

print("Fairness Analysis:")
print(f"{'Metric':<20} {'Majority (A=0)':>16} {'Minority (A=1)':>16} {'Gap':>8}")
print("-" * 65)
for key in ['positive_rate', 'tpr', 'fpr', 'precision', 'accuracy']:
    v0, v1 = metrics[0][key], metrics[1][key]
    print(f"{key:<20} {v0:>16.3f} {v1:>16.3f} {v0-v1:>8.3f}")

di = metrics[1]['positive_rate'] / metrics[0]['positive_rate']
print(f"\nDisparate Impact Ratio: {di:.3f} ({'PASS' if di >= 0.8 else 'FAIL'} the 80% rule)")

Fairness Analysis:
Metric                 Majority (A=0)   Minority (A=1)      Gap
-----------------------------------------------------------------
positive_rate                   0.811            0.552    0.259
tpr                             0.991            0.990    0.001
fpr                             0.013            0.024   -0.011
precision                       0.997            0.980    0.017
accuracy                        0.990            0.984    0.007

Disparate Impact Ratio: 0.681 (FAIL the 80% rule)


In [3]:
# ── Threshold Optimization for Equal Opportunity ────────────────
y_proba = model.predict_proba(X_test)[:, 1]

def find_fair_thresholds(y_true, y_proba, protected, target_tpr=None):
    """Find thresholds that equalize TPR across groups."""
    thresholds = np.linspace(0.1, 0.9, 81)
    group_thresholds = {}

    for group in [0, 1]:
        mask = (protected == group)
        yt, yp = y_true[mask], y_proba[mask]
        best_th, best_diff = 0.5, np.inf
        target = target_tpr or (y_true[protected==0].mean())
        for th in thresholds:
            pred = (yp >= th).astype(int)
            if yt.sum() > 0:
                tpr = ((pred == 1) & (yt == 1)).sum() / yt.sum()
                if abs(tpr - target) < best_diff:
                    best_diff = abs(tpr - target)
                    best_th = th
        group_thresholds[group] = best_th
    return group_thresholds

# Get TPR for majority group as target
mask0 = (A_test == 0)
tpr0 = ((y_pred[mask0] == 1) & (y_test[mask0] == 1)).sum() / y_test[mask0].sum()

fair_thresholds = find_fair_thresholds(y_test, y_proba, A_test, target_tpr=tpr0)
print(f"Equal-TPR thresholds: Group 0 = {fair_thresholds[0]:.2f}, Group 1 = {fair_thresholds[1]:.2f}")

# Apply group-specific thresholds
y_fair = np.zeros_like(y_pred)
for group, th in fair_thresholds.items():
    mask = (A_test == group)
    y_fair[mask] = (y_proba[mask] >= th).astype(int)

metrics_fair = fairness_metrics(y_test, y_fair, A_test)
print("\nAfter threshold correction:")
print(f"TPR Majority: {metrics_fair[0]['tpr']:.3f}  TPR Minority: {metrics_fair[1]['tpr']:.3f}")
di_fair = metrics_fair[1]['positive_rate'] / metrics_fair[0]['positive_rate']
print(f"New Disparate Impact Ratio: {di_fair:.3f}")

Equal-TPR thresholds: Group 0 = 0.50, Group 1 = 0.50

After threshold correction:
TPR Majority: 0.991  TPR Minority: 0.990
New Disparate Impact Ratio: 0.681


In [4]:
# ── Fairlearn: Constrained Optimization ────────────────────────
# from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds
# from fairlearn.metrics import MetricFrame, demographic_parity_difference

# base_estimator = LogisticRegression()
# constraint = EqualizedOdds()

# mitigator = ExponentiatedGradient(base_estimator, constraint)
# mitigator.fit(X_train, y_train, sensitive_features=A_train)
# y_pred_fair = mitigator.predict(X_test)

# # Evaluate
# mf = MetricFrame(
#     metrics={'accuracy': accuracy_score, 'selection_rate': selection_rate},
#     y_true=y_test, y_pred=y_pred_fair, sensitive_features=A_test
# )
# print(mf.by_group)
print("Fairlearn ExponentiatedGradient template ready")

Fairlearn ExponentiatedGradient template ready


## Additional Learning Resources

### Papers
- [Fairness Through Awareness (Dwork et al.)](https://arxiv.org/abs/1104.3913)
- [Equality of Opportunity in Supervised Learning (Hardt et al.)](https://arxiv.org/abs/1610.02413)
- [Inherent Trade-Offs in Algorithmic Fairness (Chouldechova)](https://arxiv.org/abs/1707.00046)

### Books & Courses
- [Fairness and Machine Learning (Barocas, Hardt, Narayanan)](https://fairmlbook.org/) Free online
- [Responsible AI Practices Google](https://ai.google/responsibilities/responsible-ai-practices/)
- [AI Fairness 360 IBM](https://aif360.mybluemix.net/)

### Tools
- [Fairlearn Docs](https://fairlearn.org/)
- [AIF360 Docs](https://aif360.readthedocs.io/)
- [What-If Tool Google](https://pair-code.github.io/what-if-tool/)
- [Aequitas Bias Audit Toolkit](http://aequitas.dssg.io/)